# RAG-Fusion with RAGAS

**Key Features**:
- RAG-Fusion implementation  
- RAGAS evaluation with LangSmith tracing

In [ ]:
import os
import hashlib
import getpass
from typing import List, Dict
from collections import defaultdict
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.output_parsers import PydanticOutputParser

from langsmith import Client, traceable
from ragas import evaluate, RunConfig, EvaluationDataset
from ragas.metrics import Faithfulness, ResponseRelevancy



#### Setup

In [19]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "RAG-Fusion"
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
ls_client = Client()

print("✅ Dependencies loaded successfully!")

✅ Dependencies loaded successfully!


#### Load and process data

In [ ]:

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

documents = loader.load()

for doc in documents:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

print(f"✅ Loaded {len(documents)} documents")

✅ Loaded 825 valid documents


#### Setup embeddings and vector store

In [25]:

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Qdrant.from_documents(
    documents, 
    embeddings, 
    location=":memory:", 
    collection_name="RAGFusion"
)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 7})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

print("✅ Vector store and LLM initialized")

✅ Vector store and LLM initialized


#### Query generation

In [26]:
class QueryVariations(BaseModel):
    direct_reformulation: str = Field(description="Direct reformulation of the question")
    specific_details: str = Field(description="Query focusing on specific details or examples")
    causes_factors: str = Field(description="Query about causes or underlying factors")
    consequences_implications: str = Field(description="Query about consequences or implications")


QUERY_GENERATION_TEMPLATE = """
You are a helpful assistant that generates multiple search queries based on a single input query.
Your goal is to generate diverse queries that capture different aspects and perspectives of the original question.

Original query: {question}

Generate 4 different search queries in the following format:

direct_reformulation: [direct reformulation of the question]
specific_details: [query focusing on specific details or examples]
causes_factors: [query about causes or underlying factors]
consequences_implications: [query about consequences or implications]

{format_instructions}
"""

query_generation_prompt = ChatPromptTemplate.from_template(QUERY_GENERATION_TEMPLATE)
parser = PydanticOutputParser(pydantic_object=QueryVariations)

@traceable
def generate_fusion_queries(question: str) -> List[str]:
    """Generate queries for fusion"""
    response = (
        query_generation_prompt 
        | llm 
        | parser
    ).invoke({
        "question": question, 
        "format_instructions": parser.get_format_instructions()
    })
    
    queries = [
        question, # include original
        response.direct_reformulation,
        response.specific_details,
        response.causes_factors,
        response.consequences_implications, 
    ]
    
    return queries[:4]

# Test query generation
test_queries = generate_fusion_queries("What are common student loan issues?")
print(f"✅ Generated {len(test_queries)} queries")
for i, q in enumerate(test_queries):
    print(f"  {i+1}. {q}")

✅ Generated 4 queries
  1. What are common student loan issues?
  2. What are the most common issues faced by students with loans?
  3. What are examples of common problems students encounter with their loans?
  4. What factors contribute to student loan issues?


#### Reciprocal Rank Fusion

In [27]:
@traceable
def reciprocal_rank_fusion(results_list: List[List[Document]], k: int = 60) -> List[Document]:
    fusion_scores = defaultdict(float)
    document_map = {}
    
    for result_list in results_list:
        for rank, doc in enumerate(result_list):
            doc_id = hashlib.sha256(doc.page_content.encode()).hexdigest()[:16]
            fusion_scores[doc_id] += 1 / (rank + k)
            document_map[doc_id] = doc
    
    # Sort by fusion score
    ranked_docs = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)
    return [document_map[doc_id] for doc_id, _ in ranked_docs[:8]]

@traceable
def rag_fusion_retrieval(question: str) -> List[Document]:
    queries = generate_fusion_queries(question)
    all_results = [base_retriever.invoke(query) for query in queries]
    return reciprocal_rank_fusion(all_results)

# Test retrieval
test_docs = rag_fusion_retrieval("What are common payment problems?")
print(f"✅ Retrieved {len(test_docs)} documents via RAG-Fusion")

✅ Retrieved 8 documents via RAG-Fusion


#### RAG Answer Generation

In [28]:
RAG_FUSION_TEMPLATE = """
You are a helpful and knowledgeable assistant. Use the provided context to answer the user's question comprehensively.

The context has been gathered using advanced retrieval techniques to ensure relevance and completeness.
Please provide a detailed and accurate answer based on the information provided.
You must only use the provided context, and cannot use your own knowledge

If you cannot find sufficient information in the context to answer the question, say "I don't know".

Context:
{context}

Question: {question}

Answer:
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_FUSION_TEMPLATE)

def format_context(docs: List[Document]) -> str:
    """Format documents for context"""
    return "\n\n".join([f"Document {i+1}: {doc.page_content}" for i, doc in enumerate(docs)])

@traceable
def rag_fusion_answer(question: str) -> Dict:
    """Generate answer using RAG-Fusion and return format for RAGAS"""
    retrieved_docs = rag_fusion_retrieval(question)
    context = format_context(retrieved_docs)
    
    answer = (rag_prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question
    })
    
    return {
        "answer": answer,
        "contexts": [doc.page_content for doc in retrieved_docs],
        "question": question
    }

# Test answer generation
test_result = rag_fusion_answer("What are common payment problems?")
print(f"✅ Generated answer with {len(test_result['contexts'])} context documents")
print(f"Answer preview: {test_result['answer'][:500]}...")

✅ Generated answer with 8 context documents
Answer preview: Common payment problems reported by users include:

1. **Payment Reversals**: Many users have experienced their payments being reversed without notification. This has occurred multiple times for various amounts, leading to confusion and frustration as payments that were confirmed as received were later not reflected in the payment history.

2. **Missing Payments**: Users have reported that payments they made did not appear in their account activity or payment history. This includes instances whe...


#### Create evaluation dataset

In [29]:

evaluation_questions = [
    "What are common student loan payment issues?",
    "How do companies handle loan complaints?", 
    "What problems occur during loan servicing?",
    "Which loan issues take longest to resolve?"
]

# Generate evaluation data
eval_data = []
for question in evaluation_questions:
    result = rag_fusion_answer(question)
    eval_data.append({
        "user_input": question,
        "response": result["answer"],
        "retrieved_contexts": result["contexts"]
    })

dataset = EvaluationDataset.from_list(eval_data)
print(f"✅ Created evaluation dataset with {len(eval_data)} examples")

✅ Created evaluation dataset with 4 examples


#### Run RAGAS evaluation with LangSmith tracing

In [32]:

print("🔄 Running RAGAS evaluation...")

custom_run_config = RunConfig(timeout=360)

ragas_result = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(),
        ResponseRelevancy(),
    ],
    llm=llm,
    embeddings=embeddings,
    run_config=custom_run_config
)

print("✅ RAGAS evaluation completed!")
print(f"📊 Results:")
print(ragas_result)

🔄 Running RAGAS evaluation...


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

✅ RAGAS evaluation completed!
📊 Results:
{'faithfulness': 0.7500, 'answer_relevancy': 0.9425}


In [33]:
@traceable(name="rrf_evaluation")
def evaluate_rrf(question: str) -> Dict:
    """RRF vs Traditional retrieval"""
    
    # Traditional retrieval
    traditional_docs = base_retriever.invoke(question)
    
    # RRF retrieval  
    rrf_docs = rag_fusion_retrieval(question)
    
    traditional_ids = set(doc.page_content[:50] for doc in traditional_docs)
    rrf_ids = set(doc.page_content[:50] for doc in rrf_docs)
    
    overlap = len(traditional_ids.intersection(rrf_ids))
    rrf_unique = len(rrf_ids - traditional_ids)
    
    return {
        "question": question,
        "traditional_docs": len(traditional_docs),
        "rrf_docs": len(rrf_docs), 
        "overlap": overlap,
        "rrf_found_new": rrf_unique,
        "diversity_score": rrf_unique / len(traditional_docs),
        "top_rrf_doc": rrf_docs[0].page_content[:100] if rrf_docs else "None"
    }

# Test on a few questions
for q in ["What are payment issues?", "How do companies respond?"]:
    result = evaluate_rrf(q)
    print(f"📊 {q}")
    print(f"   RRF found {result['rrf_found_new']} new docs (diversity: {result['diversity_score']:.2f})")

📊 What are payment issues?
   RRF found 1 new docs (diversity: 0.14)
📊 How do companies respond?
   RRF found 4 new docs (diversity: 0.57)
